In [1]:
%pip install dotenv

Note: you may need to restart the kernel to use updated packages.


In [2]:
# 라이브러리 가져오기
import requests
import pandas as pd
import os
import dotenv

In [3]:
dotenv.load_dotenv()
my_key = os.getenv('KAKAO_RESTFULL_KEY')

In [4]:
# Kakao Map API 통해 위도, 경도 데이터 가져오는 함수를 정의
def get_geocoding(place):
    try:
        url = f"https://dapi.kakao.com/v2/local/search/address"
        headers = {"Authorization": f"KakaoAK {my_key}"}
        params = {
            'query' : place
        }
        response = requests.get(url, headers=headers,params=params)
        result = response.json()
        if result["documents"]:
            return result["documents"][0]["y"], result["documents"][0]["x"]
        else:
            return None,None
    except:
        return None,None

In [5]:
url = f"https://dapi.kakao.com/v2/local/search/address"
headers = {"Authorization": f"KakaoAK {my_key}"}
params = {
    'query' : '서울특별시 종로구 상하동'
}
response = requests.get(url, headers=headers,params=params)
result = response.json()
result

{'documents': [],
 'meta': {'is_end': True, 'pageable_count': 0, 'total_count': 0}}

In [6]:
lat = []  #위도
lng = []  #경도

# 장소(주소) 리스트
places = ["서울특별시 종로구 세종대로 175", 
          "서울특별시 서초구 서초동 700", 
          "부산광역시 해운대구 해운대해변로 264"]

i=0
for place in places:   
    i = i + 1
    try:
        print(i, place)
        # get_geocoding 함수의 리턴값 호출하여 geo_location 변수에 저장
        place_lat, place_lon = get_geocoding(place)
        lat.append(place_lat)
        lng.append(place_lon)
        
    except:
        lat.append('')
        lng.append('')

# 데이터프레임으로 변환하기
df = pd.DataFrame({'위도':lat, '경도':lng}, index=places)

df

1 서울특별시 종로구 세종대로 175
2 서울특별시 서초구 서초동 700
3 부산광역시 해운대구 해운대해변로 264


,위도,경도
서울특별시 종로구 세종대로 175,37.5718478584908,126.976168275947
서울특별시 서초구 서초동 700,37.4810862955299,127.015245160054
부산광역시 해운대구 해운대해변로 264,35.1591069824231,129.160283786856


In [7]:
df = pd.read_csv('https://raw.githubusercontent.com/pia222sk20/python/main/data/sample.csv'
                 ,encoding='cp949')
df_simple = df.loc[:,['상호명','도로명주소']]
df_simple.head()

,상호명,도로명주소
0,하나산부인과,경기도 안산시 단원구 달미로 10
1,타워광명내과의원,서울특별시 강남구 언주로30길 39
2,조정현신경외과의원,경기도 시흥시 중심상가로 178
3,한귀원정신과의원,부산광역시 수영구 수영로 688
4,더블유스토어수지점,경기도 용인시 수지구 문정로 32


In [8]:
#위도 경고를 추가  apply
# df_simple['위경도'] = df_simple['도로명주소'].apply(lambda x : get_geocoding(x))
df_simple['도로명주소'].index

RangeIndex(start=0, stop=91335, step=1)

In [9]:
import random
radom_index = random.sample(range(91335),50)
radom_index[:10]

[64757, 33380, 51160, 82720, 87469, 1519, 86511, 20676, 40964, 16688]

In [ ]:

df_simple = df_simple.dropna() # 없는 데이터를 

In [11]:
df_simple_50 =  df_simple.iloc[radom_index]
results = [ get_geocoding(addr) for addr in df_simple_50['도로명주소'].values]
results[:10]

[('36.2681767871839', '127.2649069067'),
 ('37.7732128732563', '126.756751252201'),
 (None, None),
 ('37.4902539794071', '126.850433351461'),
 ('37.8516990926183', '127.744986955082'),
 ('37.8186127638268', '127.046369290345'),
 ('37.5733020510431', '127.057022081587'),
 ('37.6445861423491', '127.010519330934'),
 ('35.23331415014', '128.881774879559'),
 ('37.5128109977461', '127.021653916262')]

In [12]:
df_simple_50.loc[:,'위도'], df_simple_50.loc[:,'경도'] = zip(*results)

C:\Users\Playdata2\AppData\Local\Temp\ipykernel_16160\497364495.py:1: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df_simple_50.loc[:,'위도'], df_simple_50.loc[:,'경도'] = zip(*results)
C:\Users\Playdata2\AppData\Local\Temp\ipykernel_16160\497364495.py:1: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df_simple_50.loc[:,'위도'], df_simple_50.loc[:,'경도'] = zip(*results)


In [13]:
df_simple_50.head()

,상호명,도로명주소,위도,경도
64757,계룡이사랑치과,충청남도 계룡시 두마면 사계로 46,36.2681767871839,127.2649069067
33380,DR.장종합동물병원,경기도 파주시 풀무골로 18,37.7732128732563,126.756751252201
51160,보리한의원,부산광역시 강서구 신호산단2로49번길 16,None,None
82720,간석참조은약국,서울특별시 구로구 개봉로15길 97,37.4902539794071,126.850433351461
87469,주안내과,강원도 춘천시 퇴계농공로 15,37.8516990926183,127.744986955082


In [14]:
# 지도
%pip install folium

Note: you may need to restart the kernel to use updated packages.


In [15]:
for i in df_simple_50:
    print(i)    

상호명
도로명주소
위도
경도


In [16]:
for idx, row in df_simple_50.iterrows():
    print(idx, row['위도'], row['경도'])
    break

64757 36.2681767871839 127.2649069067


In [20]:
import folium

m = folium.Map(location=(35.8425738690814, 127.140054344904))  # 중심 좌표

# 위도, 경도가 None이 아닌 경우만 반복
for idx, row in df_simple_50.dropna(subset=['위도', '경도']).iterrows():
    folium.Marker(
        location=[row['위도'], row['경도']],
        popup=row['상호명'],
        tooltip=row['상호명'][-2:]
    ).add_to(m)

m.save('map.html')

In [19]:
# ^
# import folium

# m = folium.Map(location=(35.8425738690814, 127.140054344904))  # 최초지도 로드할때. 중심좌표
# for idx, row in df_simple_50.iterrows():
#     folium.Marker(
#         location=[ row['위도'], row['경도'] ],
#         popup=row['상호명'],
#         tooltip=row['상호명'][-2:]
#     ).add_to(m)    

# m.save('map.html')

ValueError: Location should consist of two numerical values, but None of type <class 'NoneType'> is not convertible to float.

In [ ]:
# # 지도 객체 생성
# m = folium.Map(location=[35.8425738690814, 127.140054344904], zoom_start=15) # 최초지도 로드할때, 중심좌표

# # 마커 추가
# folium.Marker(
#     location=[35.8425738690814, 127.140054344904],
#     popup='고려마취통증의학과의원',
#     tooltip='병원'
# ).add_to(m)
# m.save('map.html')
# # 지도 저장